
# 📊 Latin America Urban Mobility & Economic Analysis

## Introducción

En este proyecto analicé la relación entre la **movilidad urbana y la productividad económica en algunas de las principales ciudades de Latinoamérica**.

Para realizar el análisis trabajé con datos de **TomTom Traffic Index** y **OECD Cities**, los cuales limpié, transformé y combiné para comparar indicadores de tráfico con variables económicas e identificar qué ciudades presentan mayores oportunidades de mejora en infraestructura de transporte.

## 🧩 Paso 1: Carga y exploración

Antes de realizar la limpieza y combinación de los datos, se llevó a cabo una revisión inicial de ambos datasets para comprender su estructura, validar la correcta carga de la información e identificar posibles inconsistencias.

### 1.1 Carga de datos y vista rápida

Se importaron las librerías necesarias para el análisis:

`pandas`, `numpy`, `seaborn` y `matplotlib.pyplot`.

Posteriormente, se cargaron los datasets de **TomTom Traffic Index** y **OECD City Economy** en los DataFrames `traffic` y `eco`.

Finalmente, se revisaron las primeras filas de cada dataset para identificar sus columnas, tipos de variables y estructura general antes de comenzar con el proceso de limpieza y transformación.

In [14]:
# importar librerías
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# cargar archivos
traffic = pd.read_csv('/datasets/tomtom_traffic.csv')
eco = pd.read_csv('/datasets/oecd_city_economy.csv')

In [ ]:
# mostrar las primeras 5 filas de traffic
traffic.head(5)


In [ ]:
# mostrar las primeras 5 filas de eco
traffic.head(5)


**Tip:** Si no usas `print()` la tabla se vera mejor.

## 🧩 Paso 2: Explorar, limpiar y preparar los datos

Antes de combinar ambos datasets, se realizó una revisión de su estructura, tipos de datos, nombres de columnas y valores faltantes para identificar posibles inconsistencias y preparar la información para el análisis.

### 2.1 Explorar la estructura y tipos de datos

Se utilizó `.info()` para revisar la estructura de ambos DataFrames, incluyendo:

- Tipos de datos de cada columna
- Cantidad de valores no nulos
- Posibles valores faltantes
- Columnas que requerían conversión de tipo

También se revisaron las primeras filas de cada dataset para validar el contenido y detectar posibles problemas de formato.

A partir de esta exploración se identificaron las columnas que requerían limpieza, conversión o estandarización antes de continuar con la integración de los datos.

In [ ]:
# Examinar la estructura de traffic


traffic.info()
traffic.head(3)


traffic.isnull().sum()





En la estructura del DF traffic, se observa que:
- Las columnas `UpdateTimeUTC` y `UpdateTimeUTC` son de tipo object y deberian de ser datatime
- jamscount es float pero debería de ser integer. esto si puede afectar operaciones
-  UpdateTimeUTCWeekAgo  es object deberia de ser datatime

. no parecen haber nulos

In [ ]:
# Examinar la estructura de eco
eco.info()
eco.head(3)
eco.isnull().sum()


En la estructura del DF eco, se observa que:
- Las columnas `City GDP/capita`, `Unemployment %`, ...
- todas las columnas estan en mal tipo excepto Year
-
Year	INTEGER
City	STRING
Country	STRING
City GDP/capita	FLOAT
Unemployment %	FLOAT
PM2.5 (μg/m³)	FLOAT
Population (M)	FLOAT



### 2.2 Renombrar columnas

**🎯Objetivo:**
Estandarizar los nombres de columnas para evitar errores y facilitar la unión de los datasets.

**Instrucciones:**

- Se cambia los nombres de las columnas para que tengan el formato `snake_case`.
    - `Country` → `country`
    - `UpdateTimeUTC` → `update_time_utc`
- Verifica que los cambios se hayan aplicado correctamente usando `.columns`.





In [ ]:
traffic_clean=traffic.copy()
eco_clean=eco.copy()

In [ ]:


# Estandarizar los nombres de las columnas de traffic

traffic_clean = traffic_clean.rename(columns={
    "Country": "country",
    "City": "city",
    "UpdateTimeUTC": "update_time_utc",
    "JamsDelay": "jams_delay",
    "TrafficIndexLive": "traffic_index_live",
    "JamsLengthInKms": "jams_length_in_kms",
    "JamsCount": "jams_count",
    "TrafficIndexWeekAgo": "traffic_index_week_ago",
    "UpdateTimeUTCWeekAgo": "update_time_utc_week_ago",
    "TravelTimeLivePer10KmsMins": "travel_time_live_per10_kms_mins",
    "TravelTimeHistoricPer10KmsMins": "travel_time_historic_per10_kms_mins",
    "MinsDelay": "mins_delay"
})


# verificar cambios
traffic_clean.columns



In [ ]:


# Estandarizar los nombres de las columnas de eco
#tu código aquí
eco_clean = eco_clean.rename(columns={
    "Year": "year",
    "City": "city",
    "Country": "country",
    "City GDP/capita": "city_gdp_per_capita",
    "Unemployment %": "unemployment_percent",
    "PM2.5 (μg/m³)": "pm25_ug_m3",
    "Population (M)": "population_m"
})




# verificar cambios
eco_clean.columns


### 2.3 Corregir formatos numéricos y de fecha

**🎯Objetivo:**
Asegurar que las columnas de fechas y valores numéricos estén en formatos correctos para permitir análisis, cálculos y comparaciones precisas.

**Pasos:**

- Se convierte las columnas de fecha de `traffic` a formato `datetime`. Haz el cambio a prueba de errores.
- En el dataset `eco`, limpia los valores numéricos:
    - En `city_gdp_capita`: elimina separadores de miles (`.`) y reemplaza las comas (`','`) por puntos (`'.'`) antes de convertir a tipo `float`.
    - En `unemployment_pct`: elimina el símbolo de porcentaje (`%`) y reemplaza las comas (`','`) por puntos (`'.'`) antes de convertir a tipo `float`.
    - En `population_m`: reemplaza las comas (`','`) por puntos (`'.'`) antes de convertir a tipo `float`.
- Finalmente, crea una nueva columna llamada `population` multiplicando `population_m` por 1,000,000 para obtener la población total.


In [ ]:

# Convertir las columnas de traffic a tipo fecha con pd.to_datetime()

traffic_clean['update_time_utc'] = pd.to_datetime(traffic_clean['update_time_utc'])

traffic_clean['update_time_utc_week_ago'] = pd.to_datetime(
    traffic_clean['update_time_utc_week_ago']
)

traffic_clean.info()



In [ ]:
eco_clean.info()

In [ ]:



# Limpia separadores y convierte columnas numéricas en eco
eco_clean['city_gdp_per_capita'] = eco_clean['city_gdp_per_capita'].astype(str).str.replace('.', '').str.replace(',', '.').astype(float)
eco_clean['unemployment_percent'] = eco_clean['unemployment_percent'].astype(str).str.replace('.', '').str.replace(',', '.').str.replace('%', '').astype(float)
eco_clean['population_m'] = eco_clean['population_m'].astype(str).str.replace('.', '').str.replace(',', '.').astype(float)

# Calcula la población total en unidades absolutas (Multiplica * 1000000)

# Calcula la población total en unidades absolutas (Multiplica * 1000000)
# eco['population'] = ...
eco_clean['population'] = eco_clean['population_m'] * 1000000
# verificar el cambio
eco_clean.info()
eco_clean.head(3)




---

## 🧩Paso 3: Extraer año y filtrar

Extraer el año permite filtrar la información y trabajar solo con el período más reciente y relevante.

### 3.1 Extraer columna año y filtrar 2024

**🎯Objetivo**
Identificar el año de cada registro y mantener solo los registros del 2024.

**Pasos**

- Como el DataFrame `traffic` no tiene una columna de año, utiliza el atributo `.dt.year` sobre su columna de fecha para crear una nueva columna llamada `year`.
- Filtra las filas donde el año sea **2024**.
- Utiliza `.copy()` para crear dos nuevos DataFrames (`traffic_2024` y `eco_2024`) para evitar modificar el dataset original.

In [ ]:
# Extraer el año de las fechas en update_time_utc
traffic_2024= traffic_clean.copy()
eco_2024= eco_clean.copy()


traffic_2024['year'] = traffic_2024['update_time_utc'].dt.year


# Verificar cambio
traffic_2024.head(3)
eco_2024.head(3)

In [ ]:
# Filtra los registros del año 2024

traffic_2024 = traffic_2024[traffic_2024["year"] == 2024].copy()
eco_2024 = eco_2024[eco_2024["year"]==2024].copy()


# Revisar dataframes nuevos
display(traffic_2024.head())
display(eco_2024.head())



---

## 🧩Paso 4: Analizar y resumir datos de movilidad

Como el dataset de tráfico contiene **múltiples registros por ciudad**. En esta parte, calcularás los promedios anuales por ciudad para simplificar el análisis y obtener una visión más clara de las tendencias generales.

### 4.1 Calcular promedios de tráfico por ciudad

**🎯Objetivo:**
Obtener una vista consolidada del tráfico promedio por ciudad y año, para analizar patrones generales sin depender de datos diarios.

**Instrucciones**

- Agrupa los datos por `city`, `country` y `year`.
- Calcula el promedio **solo de las métricas de tráfico más relevantes**: como `jams_delay`, `traffic_index_live`, `jams_length_kms`, `jams_count`, `mins_delay`, y tiempos de viaje (`travel_time_live_per_10kms_mins` y `travel_time_hist_per_10kms_mins`).
- Guarda el resultado como `traffic_city_year_2024`, mantén las columnas como variables (no índices).


In [ ]:
# Calcular los  promedios de trafico por ciudad, país y año


traffic_city_year_2024 = (
    traffic_2024
        .groupby(['city', 'country', 'year'])[
            ['jams_delay',
             'traffic_index_live',
             'jams_length_in_kms',
             'jams_count',
             'mins_delay',
             'travel_time_live_per10_kms_mins',
             'travel_time_historic_per10_kms_mins']
        ]
        .mean()
        .reset_index()
)




# Mostrar resultado

traffic_city_year_2024.head()

### 🧠 **Momento de reflexión**


Ahora que ya tienes los promedios anuales por ciudad, es momento de **observarlos** con atención.

Piensa:

- ¿Cuál crees que tiene el mayor tiempo promedio de tráfico?
- ¿Será una ciudad de **Europa**, de **Latinoamérica** o de **otra región** del mundo?

Para descubrirlo, se ejecuta esta línea de código:

`traffic_city_year_2024.sort_values(["jams_delay"], ascending=False)`


🔍 Observa qué ciudad aparece en los primeros lugares.

¿Te sorprenden los resultados? , ¿Coinciden con lo que imaginabas?

In [ ]:

traffic_city_year_2024.sort_values(["jams_delay"], ascending=False)

La ciudad con el mayor tiempo promedio de tráfico es ... Mexico city


---

## 🧩Paso 5: Unir movilidad y economía

Combinar datasets te permite analizar cómo se relacionan los indicadores económicos con los de movilidad.

### 5.1 Unir tráfico (tabla principal) con indicadores económicos

**🎯Objetivo:**
Combinar la información de tráfico y economía en un solo DataFrame para analizar cómo las condiciones económicas se relacionan con la movilidad urbana.

**Pasos**
- Selecciona solo las **columnas relevantes** de cada dataset (por ejemplo, variables clave de tráfico y de economía).
- Usa `.copy()` al crear subconjuntos para evitar modificar el dataset original.
- Une ambos DataFrames y define como **claves de unión** a `city` y `year`.
- Mantén solo las ciudades y años presentes en ambos datasets.
- Guarda el resultado en una nueva variable llamada `merged` y muestra las primeras 5 filas.


In [ ]:

# Seleccionar columnas clave de tráfico y economía
left_cols = ['city','country','year','jams_delay','traffic_index_live',
             'jams_length_in_kms','jams_count','mins_delay',
             'travel_time_live_per10_kms_mins','travel_time_historic_per10_kms_mins']

right_cols = ['city','year','city_gdp_per_capita','unemployment_percent','pm25_ug_m3','population']

# Usar .copy() para crear los dos nuevos datasets reducidos
traffic_2024_small = traffic_city_year_2024[ left_cols ].copy()
eco_2024_small =  eco_2024[right_cols].copy()

# Unir datasets
merged = traffic_2024_small.merge(
    eco_2024_small,
    on=['city', 'year'],
    how='inner'
)



merged.head(5)





---

## 🧩Paso 6: Visualización y análisis de relaciones

Ahora que tienes un dataset limpio y unificado, es momento de **visualizar patrones**.
Los gráficos te ayudarán a entender cómo se relacionan las variables económicas con las de movilidad urbana.

### 6.1 Visualizar relaciones entre economía y tráfico

**🎯Objetivo:**
Analizar visualmente la distribución y la relación entre indicadores de tráfico y economía en 2024, para identificar posibles patrones o tendencias generales entre ambas variables.

**Pasos**
- Usa las librerías `seaborn` y `matplotlib.pyplot` para generar los gráficos.
- Visualiza la distribución del **tráfico** (`jams_delay`) mediante:
    - **Boxplot** → para observar la media, mediana y detectar valores atípicos.
- Visualiza la distribución de la **economía** (`city_gdp_capita`) mediante:
    - **Histograma** → para analizar la forma de la distribución y el valor promedio del PIB per cápita.
- Finalmente, **compara ambas variables**, para observar si existe alguna relación entre ellas, haciendo un solo gráfico de barras donde aparezcan ambos indicadores.
- Recuerda agregar título y etiquetas a los ejes de tus gráficos.
- Observa y comenta los patrones, valores extremos o posibles relaciones que identifiques.

In [ ]:

# Crear boxplot para observar el comportamiento de los minutos de congestion JamsDelay
# crea tu gráfico
sns.boxplot(data=merged, x='country', y='jams_delay', showmeans=True )
# obtener promedio para mostrarlo en título
mean_value = merged['jams_delay'].mean()
plt.title(f'Boxplot de JamsDelay (2024)\nPromedio: {mean_value:.2f}')
plt.figure(figsize=(10, 2))
plt.show()



In [ ]:
# Crear histograma para ver la distribución de la economía (city_gdp_capita)

plt.hist(merged['city_gdp_per_capita'], bins=20)
plt.xlabel('city_gdp_per_capita')
plt.ylabel('Frecuencia')
plt.title('Distribución de city_gdp_capita')
plt.xticks(rotation=90)
plt.show()

<div class="alert alert-block alert-success">
<b>Comentario #1</b> <a class="tocSkip"></a>

Se Generó correctamente el histograma de PIB per cápita y se añadió etiquetas adecuadas. Esta visualización es útil para entender la distribución de la variable, detectar sesgos y evaluar la presencia de valores atípicos.

</div>

In [ ]:
eco_2024.info()

In [ ]:

# Opción 1: Gráfico básico corregido
merged.plot(kind="bar", x="city", y=['jams_delay', 'city_gdp_per_capita'])
plt.xticks(rotation=90)
plt.show()

### 🧠 **Reflexiona**
Excelente trabajo llegando a esta etapa del análisis. Antes de avanzar, revisa tus gráficos, tómate un momento para pensar:

* ¿Las ciudades con mayor PIB per cápita también presentan más congestión?

* ¿O sucede lo contrario, o no existe una relación clara?

Comentarios: No, no parecen ser proporcionales. al menos no en la mayoria de casos ya que son datos muy diferentes. por ejemplo ciudad de mexico clasisifca con el mayor trafrico de las ciudades y aunque su GDP es similar por otro lado monte video tiene un trafico minimo y su GDP es incluso mas grande que el de ciudad de mexico


---

## 🧩Paso 7: Exportar y documentar resultados

En esta etapa final consolidarás todo tu trabajo: guardarás el dataset limpio y crearás un resumen que documente los resultados del proyecto.

### 7.1 Guardar dataset final

**🎯Objetivo:**
Generar un CSV limpio, reproducible y con columnas relevantes para análisis posterior.

**Instrucciones**

- Exporta el DataFrame `merged` con el nombre: `ladb_mobility_economy_2024_clean.csv`
- Usa `index=False` para no incluir el índice.


In [ ]:
# Exporta el dataset final como CSV
merged.to_csv("ladb_mobility_economy_2024_clean.csv", index=False)

Para poder ver o descargar el archivo generado:   
En el menú lateral que esta a la izquierda, ve hasta la parte de abajo, a la sección de **Exportar dataset** para más información.


---

# 🧾 Resumen ejecutivo
> Completa este resumen al finalizar el análisis. Mantén 3–5 párrafos cortos, claros y accionables.

**Contexto & objetivo:**  
- Responde la pregunta central del análisis: ¿qué relación existe entre la movilidad urbana (congestión, tiempos de viaje) y la productividad económica (PIB per cápita)?
- Explica brevemente las variables clave utilizadas y su relevancia para la toma de decisiones.


*R:* El análisis buscó responder una pregunta clave: ¿existe una relación entre la congestión urbana y la productividad económica (PIB per cápita)? Para ello se utilizaron indicadores de tráfico como traffic_index_live, jams_delay, mins_delay y tiempos promedio de viaje, junto con variables económicas como city_gdp_per_capita, desempleo y población. La intención es entender si mayores niveles de congestión están asociados con menor productividad, lo cual puede orientar decisiones de inversión en infraestructura.

**Cobertura de datos:**  
- Especifica los años analizados, número de ciudades y países incluidos.
 *R:* El análisis incluye 15 ciudades de América Latina correspondientes al año 2024. Todas las observaciones fueron consolidadas a nivel ciudad–año para evitar ruido diario y facilitar comparaciones estructurales entre economías urbanas.

**Metodología (alto nivel):**  
- Describe los procesos principales: limpieza de datos (formatos, estandarización de columnas).
- Explica la agregación por ciudad–año y el uso de una unión INNER para integrar tráfico y economía.
- Menciona las validaciones visuales empleadas (distribuciones, outliers, tendencias generales).
  
 *R:* Se realizó limpieza y estandarización de columnas, conversión de formatos numéricos (incluyendo separadores decimales), y agregación de métricas de tráfico por ciudad–año. Posteriormente se integraron los datasets de movilidad y economía mediante una unión tipo INNER para asegurar consistencia en las observaciones. Se validaron distribuciones y posibles valores atípicos mediante revisión exploratoria.

**Hallazgos iniciales:**  
- Resume los patrones más importantes entre índices de tráfico y PIB per cápita.
- Destaca anomalías u outliers que podrían requerir revisión adicional o un análisis más profundo.
  
*R:*  A nivel general, la correlación entre el índice de tráfico y el PIB per cápita es muy débil (≈ 0.05), lo que indica que no existe una relación lineal clara entre congestión y productividad económica cuando se observan todas las ciudades en conjunto.

Sin embargo, al observar casos específicos:

Bogotá presenta uno de los niveles más altos de congestión y, al mismo tiempo, un PIB per cápita inferior al de Buenos Aires.

Buenos Aires muestra menor congestión relativa y un PIB per cápita considerablemente mayor.

Lima se ubica en un punto intermedio tanto en congestión como en productividad.

Esto sugiere que Bogotá es el caso donde la combinación de alta congestión y menor productividad es más evidente dentro del grupo analizado.

**Recomendaciones**  
- Aterriza los hallazgos en acciones: ciudades prioritarias, necesidad de validar fuentes, requerimiento de análisis adicionales, o propuestas de inversión.

*R:* Ciudad prioritaria: Bogotá emerge como la ciudad donde la alta congestión coincide con indicadores económicos menos favorables en comparación con pares regionales. Podría considerarse prioritaria para inversión en infraestructura de transporte.

Validar si los datos de desempleo y PIB tienen la misma base metodológica entre países.

Complementar el análisis con datos longitudinales (más años) para evaluar tendencia y no solo fotografía 2024.

Incorporar variables adicionales como inversión pública en transporte o uso de transporte masivo para profundizar el diagnóstico.

**Pregunta**
- ¿Qué ciudad : Bogotá, Lima o Buenos Aires o alguna otra en particular, muestra la mayor correlación significativa entre altos niveles de congestión vehicular y bajos indicadores de productividad económica, sugiriendo ser una ciudad prioritaria para inversión en infraestructura de transporte?

*R:* Entre Bogotá, Lima y Buenos Aires, Bogotá muestra la señal más clara de alta congestión combinada con menor productividad relativa, por lo que sería la ciudad más prioritaria para evaluar inversión estratégica en infraestructura de movilidad.
